# Compensation Group Analysis — Template

Computes per-group summary statistics for compensation across each
**Job_Family × Compensation_Grade × Country** combination:

- **Average** (mean) and **median** compensation
- **Standard deviation** and **coefficient of variation** (std / mean)
- **Min**, **P25**, **P75**, **Max**, plus **IQR** and **range**
- **Sample size** (count of records in the group)

**How to use:** edit the `CONFIG` cell to point at your data file and confirm
the column-name mapping, then run all cells (Kernel ▸ Restart & Run All).

_Data source: ADM — Analytics Data Mart. Compensation field = `Hire_Pay` (USD)._

## 1. Configuration

Set the data path and map your actual column names here. Nothing below this cell should need editing.

In [ ]:
# ---- CONFIG: edit these to match your data ----

DATA_PATH = 'data.csv'        # path to ADM export (.csv, .xlsx, or .parquet)
SHEET_NAME = 0                # used only for .xlsx (sheet index or name)

# Map the logical fields used by this notebook -> the column names in YOUR data.
COLS = {
    'job_family':        'Job_Family',
    'compensation_grade':'Compensation_Grade',
    'country':           'Country',          # if you only have 'Location', map it here
    'compensation':      'Hire_Pay',         # the numeric pay column to analyze
}

# Group keys, in order. Output is one row per unique combination.
GROUP_KEYS = ['job_family', 'compensation_grade', 'country']

# Drop groups with fewer than this many records from the 'reliable' view (set to 1 to keep all).
MIN_SAMPLE_SIZE = 5

# Where to write the result tables.
OUTPUT_XLSX = 'compensation_group_summary.xlsx'
OUTPUT_CSV  = 'compensation_group_summary.csv'

## 2. Imports

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option('display.max_rows', 200)
pd.set_option('display.float_format', lambda v: f'{v:,.2f}')

## 3. Load data

Supports CSV, Excel, and Parquet based on file extension.

In [ ]:
def load_data(path, sheet_name=0):
    p = Path(path)
    if not p.exists():
        raise FileNotFoundError(
            f"Could not find '{path}'. Update DATA_PATH in the CONFIG cell.")
    ext = p.suffix.lower()
    if ext == '.csv':
        return pd.read_csv(p)
    if ext in ('.xlsx', '.xls'):
        return pd.read_excel(p, sheet_name=sheet_name)
    if ext == '.parquet':
        return pd.read_parquet(p)
    raise ValueError(f'Unsupported file type: {ext}')

df_raw = load_data(DATA_PATH, SHEET_NAME)
print(f'Loaded {len(df_raw):,} rows x {df_raw.shape[1]} columns')
df_raw.head()

## 4. Validate & prepare

Confirms mapped columns exist, coerces compensation to numeric, and reports how many rows are usable.

In [ ]:
# Check every mapped column is present.
missing = [src for src in COLS.values() if src not in df_raw.columns]
if missing:
    raise KeyError(
        f'These mapped columns are not in the data: {missing}. '
        f'Available columns: {list(df_raw.columns)}')

# Standardize to the logical names used below.
df = df_raw.rename(columns={src: logical for logical, src in COLS.items()}).copy()

# Coerce compensation to numeric; non-numeric values become NaN.
df['compensation'] = pd.to_numeric(df['compensation'], errors='coerce')

n_total = len(df)
n_missing_comp = df['compensation'].isna().sum()
n_missing_keys = df[GROUP_KEYS].isna().any(axis=1).sum()

print(f'Rows total:                       {n_total:,}')
print(f'Rows with non-numeric/missing pay:{n_missing_comp:,}')
print(f'Rows missing a group key:         {n_missing_keys:,}')

# Keep only rows usable for the analysis (valid pay AND complete group keys).
df_valid = df.dropna(subset=['compensation'] + GROUP_KEYS).copy()
print(f'Rows used in analysis:            {len(df_valid):,}')

## 5. Group summary

One row per **Job_Family × Compensation_Grade × Country** group:

- **sample_size** — records in the group
- **avg_comp / median_comp** — central tendency
- **std_comp** — standard deviation (NaN when n == 1)
- **cv_comp** — coefficient of variation (std / mean); proxy for forecast difficulty
- **min / p25 / p75 / max** — distribution, with **iqr_comp** (p75 − p25) and **comp_range** (max − min)

Edit `AGGS` to add or remove statistics.

In [ ]:
# Percentile helpers (named-agg needs a callable for quantiles).
def _p(q):
    f = lambda x: x.quantile(q)
    f.__name__ = f'p{int(q*100)}'
    return f

AGGS = {
    'sample_size': ('compensation', 'size'),
    'avg_comp':    ('compensation', 'mean'),
    'median_comp': ('compensation', 'median'),
    'std_comp':    ('compensation', 'std'),     # spread; NaN when n == 1
    'min_comp':    ('compensation', 'min'),
    'p25_comp':    ('compensation', _p(0.25)),
    'p75_comp':    ('compensation', _p(0.75)),
    'max_comp':    ('compensation', 'max'),
}

def summarize(data, keys, aggs):
    out = (data
           .groupby(keys, dropna=False)
           .agg(**aggs)
           .reset_index()
           .sort_values(keys)
           .reset_index(drop=True))
    out['comp_range'] = out['max_comp'] - out['min_comp']
    out['iqr_comp']   = out['p75_comp'] - out['p25_comp']        # robust spread
    # Coefficient of variation: std / mean. Directly relevant to pay-rate
    # variance / expected MAPE — high CV groups are harder to forecast.
    out['cv_comp']    = out['std_comp'] / out['avg_comp']
    # Order columns: keys, then stats grouped logically.
    stat_order = ['sample_size','avg_comp','median_comp','std_comp','cv_comp',
                  'min_comp','p25_comp','p75_comp','max_comp','iqr_comp','comp_range']
    return out[keys + [c for c in stat_order if c in out.columns]]

summary = summarize(df_valid, GROUP_KEYS, AGGS)
print(f'{len(summary):,} unique groups')
summary

## 6. Reliable groups only

Groups meeting the `MIN_SAMPLE_SIZE` threshold. Small-sample groups have noisy statistics, so segment them out for forecasting.

In [ ]:
reliable = summary[summary['sample_size'] >= MIN_SAMPLE_SIZE].reset_index(drop=True)
small    = summary[summary['sample_size'] <  MIN_SAMPLE_SIZE].reset_index(drop=True)

print(f'Reliable groups (n >= {MIN_SAMPLE_SIZE}): {len(reliable):,}')
print(f'Small-sample groups:        {len(small):,}')
reliable

## 7. Optional — single-segment drill-down

Filter to one slice to inspect it. Set any value to `None` to ignore that dimension.

In [ ]:
def drill(data, job_family=None, compensation_grade=None, country=None):
    mask = pd.Series(True, index=data.index)
    if job_family is not None:
        mask &= data['job_family'] == job_family
    if compensation_grade is not None:
        mask &= data['compensation_grade'] == compensation_grade
    if country is not None:
        mask &= data['country'] == country
    return data[mask]

# Example (edit values to ones present in your data):
# drill(summary, country='United States')

## 8. Export

Writes the full summary and the reliable-only view to Excel (two sheets) and CSV.

In [ ]:
with pd.ExcelWriter(OUTPUT_XLSX, engine='openpyxl') as xw:
    summary.to_excel(xw, sheet_name='all_groups', index=False)
    reliable.to_excel(xw, sheet_name=f'reliable_n_ge_{MIN_SAMPLE_SIZE}', index=False)
    small.to_excel(xw, sheet_name='small_sample', index=False)

summary.to_csv(OUTPUT_CSV, index=False)
print(f'Wrote {OUTPUT_XLSX} and {OUTPUT_CSV}')